# SVM - analiza dla zbiorow Years i Wine

Notebook realizuje wymagania z prezentacji:
- wczytanie danych przez `pandas` z lokalnego katalogu,
- wybor 2 cech,
- podzial train/test = 0.30,
- analiza 4 jader SVM (`linear`, `rbf`, `poly`, `sigmoid`),
- macierze pomylek, accuracy train/test,
- ocena modelu: dopasowany / nadmiernie dopasowany / niedopasowany,
- wskazanie najlepszego jadra na koncu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')
np.random.seed(42)

base_dir = Path.cwd()
years_path = base_dir / 'VLagun_Phys_Years3.csv'
wine_path = base_dir / 'wine.csv'

print('Katalog roboczy:', base_dir)
print('Years CSV:', years_path)
print('Wine CSV:', wine_path)

In [ ]:
years_df = pd.read_csv(years_path)
wine_df = pd.read_csv(wine_path)

display(years_df.head())
display(wine_df.head())

print('Years shape:', years_df.shape)
print('Wine shape:', wine_df.shape)

In [ ]:
def classify_fit(train_acc, test_acc):
    if train_acc < 0.60 and test_acc < 0.60:
        return 'Model niedopasowany'
    if train_acc > 0.60 and test_acc > 0.60:
        if test_acc < train_acc:
            return 'Model dopasowany'
        return 'Model nadmiernie dopasowany'
    if train_acc > test_acc:
        return 'Model nadmiernie dopasowany'
    return 'Model niedopasowany'


def run_svm_for_kernels(df, feature_cols, target_col, title):
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    kernels = ['linear', 'rbf', 'poly', 'sigmoid']
    rows = []
    fitted_models = {}

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()

    for i, kernel in enumerate(kernels):
        model = SVC(kernel=kernel, random_state=42)
        model.fit(X_train_sc, y_train)

        y_train_pred = model.predict(X_train_sc)
        y_test_pred = model.predict(X_test_sc)

        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        cm_test = confusion_matrix(y_test, y_test_pred)
        sns.heatmap(
            cm_test,
            annot=True,
            fmt='d',
            cmap='YlGnBu',
            cbar=False,
            ax=axes[i]
        )
        axes[i].set_title(f'{title} - {kernel} (test)')
        axes[i].set_xlabel('Predykcja')
        axes[i].set_ylabel('Rzeczywiste')

        rows.append({
            'kernel': kernel,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'ocena_modelu': classify_fit(train_acc, test_acc)
        })
        fitted_models[kernel] = model

    plt.tight_layout()
    plt.show()

    result_df = pd.DataFrame(rows).sort_values('test_accuracy', ascending=False)
    return X_train, X_test, y_train, y_test, scaler, fitted_models, result_df


def plot_boundaries(X_train, y_train, X_test, y_test, scaler, models, title):
    kernels = ['linear', 'rbf', 'poly', 'sigmoid']
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()

    X_all = pd.concat([X_train, X_test], axis=0)
    x_min, x_max = X_all.iloc[:, 0].min() - 1, X_all.iloc[:, 0].max() + 1
    y_min, y_max = X_all.iloc[:, 1].min() - 1, X_all.iloc[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )

    mesh = np.c_[xx.ravel(), yy.ravel()]
    mesh_sc = scaler.transform(mesh)

    y_train_arr = np.array(y_train)
    y_test_arr = np.array(y_test)

    for i, kernel in enumerate(kernels):
        model = models[kernel]
        zz = model.predict(mesh_sc).reshape(xx.shape)

        axes[i].contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
        axes[i].scatter(
            X_train.iloc[:, 0], X_train.iloc[:, 1],
            c=y_train_arr,
            cmap='coolwarm',
            edgecolor='k',
            s=45,
            label='train'
        )
        axes[i].scatter(
            X_test.iloc[:, 0], X_test.iloc[:, 1],
            c=y_test_arr,
            cmap='coolwarm',
            marker='^',
            edgecolor='k',
            s=55,
            label='test'
        )
        axes[i].set_title(f'{title} - granica decyzyjna ({kernel})')
        axes[i].set_xlabel(X_train.columns[0])
        axes[i].set_ylabel(X_train.columns[1])
        axes[i].legend(loc='best')

    plt.tight_layout()
    plt.show()

## 1) Zbior Years (2 klasy: `Years` = 0, 1)
Wybrane cechy: `temp.` i `Windspeedinsitu`

In [ ]:
years_features = ['temp.', 'Windspeedinsitu']
years_target = 'Years'

(
    years_X_train,
    years_X_test,
    years_y_train,
    years_y_test,
    years_scaler,
    years_models,
    years_results
) = run_svm_for_kernels(
    years_df,
    feature_cols=years_features,
    target_col=years_target,
    title='Years'
)

display(years_results)

In [ ]:
plot_boundaries(
    years_X_train,
    years_y_train,
    years_X_test,
    years_y_test,
    years_scaler,
    years_models,
    title='Years'
)

## 2) Zbior Wine (3 klasy: `Wine` = 1, 2, 3)
Wybrane cechy: `Alcohol` i `Phenols`

In [ ]:
wine_features = ['Alcohol', 'Phenols']
wine_target = 'Wine'

(
    wine_X_train,
    wine_X_test,
    wine_y_train,
    wine_y_test,
    wine_scaler,
    wine_models,
    wine_results
) = run_svm_for_kernels(
    wine_df,
    feature_cols=wine_features,
    target_col=wine_target,
    title='Wine'
)

display(wine_results)

In [ ]:
plot_boundaries(
    wine_X_train,
    wine_y_train,
    wine_X_test,
    wine_y_test,
    wine_scaler,
    wine_models,
    title='Wine'
)

## 3) Wnioski koncowe (markdown)
W tej komorce automatycznie wybieramy jadro z najwyzszym `test_accuracy` osobno dla zbioru Years i Wine.

In [ ]:
best_years = years_results.iloc[0]
best_wine = wine_results.iloc[0]

summary_md = f"""
### Najlepsze jadra SVM

- **Years**: `{best_years['kernel']}` (test accuracy = **{best_years['test_accuracy']:.4f}**)
- **Wine**: `{best_wine['kernel']}` (test accuracy = **{best_wine['test_accuracy']:.4f}**)

### Ocena dopasowania (na bazie relacji train/test)

- **Years ({best_years['kernel']})**: {best_years['ocena_modelu']}
- **Wine ({best_wine['kernel']})**: {best_wine['ocena_modelu']}
"""

display(Markdown(summary_md))